In [44]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import re
import string

In [5]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [6]:
print(os.getcwd())

/content


In [10]:
import os
import kagglehub

# Download the dataset using kagglehub
# This will download the dataset to a local path in the Colab environment.
dataset_path = kagglehub.dataset_download('jainpooja/fake-news-detection')

# Construct the full paths to the CSV files using the downloaded dataset path
df_fake = pd.read_csv(os.path.join(dataset_path, "Fake.csv"))
df_true = pd.read_csv(os.path.join(dataset_path, "True.csv"))

100%|██████████| 41.0M/41.0M [00:00<00:00, 90.8MB/s]

Extracting files...


In [11]:
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [12]:
df_fake["class"] = 0
df_true["class"] = 1

In [13]:
df_fake.shape, df_true.shape

((23481, 5), (21417, 5))

In [14]:
# Removing last 10 rows for manual testing
df_fake_manual_testing = df_fake.tail(10)
for i in range(1, 11):
    df_fake.drop([(df_fake.shape[0] - i)], axis = 0, inplace = True)


df_true_manual_testing = df_true.tail(10)
for i in range(1, 11):
    df_true.drop([(df_true.shape[0] - i)], axis = 0, inplace = True)

In [15]:
df_fake.shape, df_true.shape

((23471, 5), (21407, 5))

In [16]:
df_fake_manual_testing.reset_index(drop=True, inplace=True)
df_true_manual_testing.reset_index(drop=True, inplace=True)

In [17]:
df_true_manual_testing

,title,text,subject,date,class
0,"Mata Pires, owner of embattled Brazil builder ...","SAO PAULO (Reuters) - Cesar Mata Pires, the ow...",worldnews,"August 22, 2017",1
1,"U.S., North Korea clash at U.N. forum over nuc...",GENEVA (Reuters) - North Korea and the United ...,worldnews,"August 22, 2017",1
2,"U.S., North Korea clash at U.N. arms forum on ...",GENEVA (Reuters) - North Korea and the United ...,worldnews,"August 22, 2017",1
3,Headless torso could belong to submarine journ...,COPENHAGEN (Reuters) - Danish police said on T...,worldnews,"August 22, 2017",1
4,North Korea shipments to Syria chemical arms a...,UNITED NATIONS (Reuters) - Two North Korean sh...,worldnews,"August 21, 2017",1
5,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",1
6,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",1
7,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",1
8,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",1
9,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017",1


In [18]:
df_manual_testing = pd.concat([df_fake_manual_testing, df_true_manual_testing], axis = 0)
# df_manual_testing.reset_index(drop=True, inplace=True)
df_manual_testing.to_csv("manual_testing.csv")

In [19]:
df_merge = pd.concat([df_fake, df_true], axis=0 )
df_merge.head(10)

,title,text,subject,date,class
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0
5,Racist Alabama Cops Brutalize Black Boy While...,The number of cases of cops brutalizing and ki...,News,"December 25, 2017",0
6,"Fresh Off The Golf Course, Trump Lashes Out A...",Donald Trump spent a good portion of his day a...,News,"December 23, 2017",0
7,Trump Said Some INSANELY Racist Stuff Inside ...,In the wake of yet another court decision that...,News,"December 23, 2017",0
8,Former CIA Director Slams Trump Over UN Bully...,Many people have raised the alarm regarding th...,News,"December 22, 2017",0
9,WATCH: Brand-New Pro-Trump Ad Features So Muc...,Just when you might have thought we d get a br...,News,"December 21, 2017",0


In [20]:
df_merge.columns

Index(['title', 'text', 'subject', 'date', 'class'], dtype='object')

In [21]:
df = df_merge.drop(["title", "subject","date"], axis = 1)
df.shape

(44878, 2)

In [22]:
df.isnull().sum()

,0
text,0
class,0


In [23]:
df = df.sample(frac = 1)
df.shape

(44878, 2)

In [24]:
df.head(10)

,text,class
21149,SEOUL (Reuters) - U.S. President Donald Trump ...,1
1377,"On Monday, Donald Trump once again embarrassed...",0
1748,"A dying man had just one wish, and that was fo...",0
21098,You re gonna love this patriot! He speaks for ...,0
16738,Stop counting the votes! Your candidates nomin...,0
4973,(Reuters) - Shares of hospitals and health ins...,1
1926,"HOUSTON (Reuters) - Thirteen Superfund sites, ...",1
5612,Even though the fact that the presumptive Repu...,0
3959,WASHINGTON (Reuters) - U.S. House Speaker Paul...,1
21026,"So far, this video has over 530,000 views. Doe...",0


In [25]:
df.reset_index(drop=True, inplace=True)

In [26]:
df.head(5)

,text,class
0,SEOUL (Reuters) - U.S. President Donald Trump ...,1
1,"On Monday, Donald Trump once again embarrassed...",0
2,"A dying man had just one wish, and that was fo...",0
3,You re gonna love this patriot! He speaks for ...,0
4,Stop counting the votes! Your candidates nomin...,0


In [27]:
def process_text(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_501/3571241116.py:3: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
/tmp/ipykernel_501/3571241116.py:5: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www\.\S+', '', text)
/tmp/ipykernel_501/3571241116.py:9: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [28]:
df["text"] = df["text"].apply(process_text)

In [29]:
x = df["text"]
y = df["class"]

In [30]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25)

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

In [32]:
from sklearn.linear_model import LogisticRegression

LR = LogisticRegression()
LR.fit(xv_train,y_train)

LogisticRegression()

In [33]:
pred_lr=LR.predict(xv_test)
LR.score(xv_test, y_test)

0.9854723707664884

In [34]:
print(classification_report(y_test, pred_lr))


              precision    recall  f1-score   support

           0       0.99      0.98      0.99      5857
           1       0.98      0.99      0.98      5363

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220



In [35]:
from sklearn.tree import DecisionTreeClassifier

DT = DecisionTreeClassifier()
DT.fit(xv_train, y_train)

DecisionTreeClassifier()

In [36]:
pred_dt = DT.predict(xv_test)
DT.score(xv_test, y_test)

0.995632798573975

In [37]:
print(classification_report(y_test, pred_dt))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      5857
           1       1.00      0.99      1.00      5363

    accuracy                           1.00     11220
   macro avg       1.00      1.00      1.00     11220
weighted avg       1.00      1.00      1.00     11220



In [38]:
from sklearn.ensemble import RandomForestClassifier

RFC = RandomForestClassifier(random_state=0)
RFC.fit(xv_train, y_train)

RandomForestClassifier(random_state=0)

In [39]:
pred_rfc = RFC.predict(xv_test)
RFC.score(xv_test, y_test)

0.9901960784313726

In [40]:
print(classification_report(y_test, pred_rfc))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5857
           1       0.99      0.99      0.99      5363

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220



In [41]:
def output_lable(n):
    if n == 0:
        return "Fake News"
    elif n == 1:
        return "Not A Fake News"

def manual_testing(news):
    testing_news = {"text":[news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(process_text)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_RFC = RFC.predict(new_xv_test)

    return print("\n\nLR Prediction: {} \nDT Prediction: {} \nRFC Prediction: {}".format(output_lable(pred_LR[0]),
                                                                                                              output_lable(pred_DT[0]),
                                                                                                              output_lable(pred_RFC[0])))

In [42]:
df_manual_testing = df_manual_testing.sample(frac = 1)
df_manual_testing.reset_index(drop=True, inplace=True)

In [43]:
news = df_manual_testing.iloc[4, :]
manual_testing(news.text), news["class"]



LR Prediction: Fake News 
DT Prediction: Fake News 
RFC Prediction: Fake News


(None, np.int64(0))